In [ ]:
"""
Ce fichier sert à nettoyer les données brutes, les fusionner en un seul dataset,
effectuer du feature engineering, et tester plusieurs modèles de machine learning.
Le Random Forest a montré la meilleure précision et est donc le modèle final choisi.
Le calcul de la target se fait via une fonction de scoring qui attribue un point
à un ticker si ses features respectent des critères d'investissement. Le seuil
de décision (ici 0.5) permet de régler la rigueur des critères. 
C’est ce modèle qui est utilisé par predict.py pour prédire si un investissement est recommandé.
"""



#organiser et fusionner des fichiers CSV de données financières
import pandas as pd
import os
from glob import glob
import numpy as np

# --- 1. Définir les dossiers où sont les fichiers CSV ---
prices_folder = "data/prices"          
fundamentals_folder = "data/fundamentals"  

# --- 2. Lire et concaténer tous les fichiers prices ---
price_files = glob(os.path.join(prices_folder, "*.csv"))

prices_list = []
for file in price_files:
    # Lire le CSV en ignorant les lignes inutiles
    df = pd.read_csv(file, skiprows=[1, 2])  # ignore Ticker et Date,,,,
    
    # Renommer la première colonne si nécessaire
    if df.columns[0] != "Date":
        df.rename(columns={df.columns[0]: "Date"}, inplace=True)
    
    # Convertir la colonne Date en datetime
    df["Date"] = pd.to_datetime(df["Date"])
    
    # Ajouter le ticker depuis le nom du fichier
    ticker = os.path.basename(file).split(".")[0]
    df["Ticker"] = ticker
    
    prices_list.append(df)

prices_df = pd.concat(prices_list, ignore_index=True)
print("Prices dataset shape:", prices_df.shape)

# --- 3. Lire et concaténer tous les fichiers fundamentals ---
fund_files = glob(os.path.join(fundamentals_folder, "*.csv"))

funds_list = []
for file in fund_files:
    # Lire le CSV en ignorant les lignes inutiles
    df = pd.read_csv(file, skiprows=[1, 2], low_memory=False)  # skip Ticker et ligne vide
    
    # Renommer la première colonne si nécessaire
    if df.columns[0] != "Date":
        df.rename(columns={df.columns[0]: "Date"}, inplace=True)
    
    # Convertir la colonne Date en datetime
    df["Date"] = pd.to_datetime(df["Date"])
    
    # Ajouter le ticker depuis le nom du fichier
    ticker = os.path.basename(file).split("_")[0]  # ex: 'AAPL_fundamentals.csv'
    df["Ticker"] = ticker
    
    funds_list.append(df)

fundamentals_df = pd.concat(funds_list, ignore_index=True)
print("Fundamentals dataset shape:", fundamentals_df.shape)

# Pour prices_df
cols = ["Ticker"] + [c for c in prices_df.columns if c != "Ticker"]
prices_df = prices_df[cols]

# Pour fundamentals_df
cols = ["Ticker"] + [c for c in fundamentals_df.columns if c != "Ticker"]
fundamentals_df = fundamentals_df[cols]

print(prices_df.head())

print(fundamentals_df.head())

# --- 4. Sauvegarder les datasets fusionnés ---
prices_df.to_csv("all_prices.csv", index=False)
fundamentals_df.to_csv("all_fundamentals.csv", index=False)


Prices dataset shape: (2776288, 8)
Fundamentals dataset shape: (2439, 334)
  Ticker       Date  Adj Close      Close       High        Low       Open  \
0      A 2000-01-03  43.113316  51.502148  56.464592  48.193848  56.330471   
1      A 2000-01-04  39.819954  47.567955  49.266811  46.316166  48.730328   
2      A 2000-01-05  37.349922  44.617310  47.567955  43.141991  47.389126   
3      A 2000-01-06  35.927761  42.918453  44.349072  41.577251   44.08083   
4      A 2000-01-07  38.921741  46.494991  47.165592  42.203148  42.247852   

      Volume  
0  4674353.0  
1  4765083.0  
2  5758642.0  
3  2534434.0  
4  2819626.0  
  Ticker       Date  Tax Effect Of Unusual Items  Tax Rate For Calcs  \
0   AAPL 2024-09-30                          0.0            0.210000   
1   AAPL 2024-12-31                          0.0            0.147000   
2   AAPL 2025-03-31                          0.0            0.155000   
3   AAPL 2025-06-30                          0.0            0.164000   
4   AA

In [4]:
#nettoyer les données:
# Supprimer la colonne Close puisque Close_Adj est plus pertinente ( elle tient compte des dividendes et fractionnements d'actions)
if "Close" in prices_df.columns:
    prices_df.drop(columns=["Close"], inplace=True)

# Réordonner les colonnes pour mettre Ticker en premier
cols = ["Ticker"] + [c for c in prices_df.columns if c != "Ticker"]
prices_df = prices_df[cols]
print(prices_df.head())


# --- Filtrer les données à partir de 2024 pour travailler sur des données recentes ---
prices_df = prices_df[prices_df["Date"] >= "2024-01-01"]
print("Filtered Prices dataset shape:", prices_df.shape)


#nettoyer fundamentals_df en gardant uniquement les colonnes essentielles pour calculer des ratios financiers
# Colonnes essentielles pour les ratios fondamentaux
cols_to_keep = [
    # Identifiants
    "Ticker", "Date",
    
    # Revenus et bénéfices
    "Total Revenue", "Operating Revenue", "EBITDA", "EBIT", "Operating Income",
    "Net Income", "Net Income From Continuing Operations", "Net Income Common Stockholders",
    
    # Actions et EPS
    "Diluted Average Shares", "Basic Average Shares", "Diluted EPS", "Basic EPS",
    
    # Bilan
    "Total Debt", "Net Debt", "Current Assets", "Current Liabilities", "Cash Cash Equivalents And Short Term Investments",
    "Accounts Receivable", "Inventory", "Invested Capital", "Total Equity", "Common Stock Equity",
    
    # Coût
    "Cost Of Revenue"
]
# Sélectionner uniquement les colonnes qui existent dans le DataFrame fundamentals_df
cols_to_keep_existing = [col for col in cols_to_keep if col in fundamentals_df.columns]

# Conserver uniquement les colonnes nécessaires
fundamentals_df_clean = fundamentals_df[cols_to_keep_existing]

fundamentals_df=fundamentals_df_clean

print("Cleaned Fundamentals dataset shape:", fundamentals_df.shape)

# --- 4. Sauvegarder  ---
prices_df.to_csv("all_prices.csv", index=False)
fundamentals_df.to_csv("all_fundamentals.csv", index=False)


  Ticker       Date  Adj Close       High        Low       Open     Volume
0      A 2000-01-03  43.113316  56.464592  48.193848  56.330471  4674353.0
1      A 2000-01-04  39.819954  49.266811  46.316166  48.730328  4765083.0
2      A 2000-01-05  37.349922  47.567955  43.141991  47.389126  5758642.0
3      A 2000-01-06  35.927761  44.349072  41.577251   44.08083  2534434.0
4      A 2000-01-07  38.921741  47.165592  42.203148  42.247852  2819626.0
Filtered Prices dataset shape: (126639, 7)
Cleaned Fundamentals dataset shape: (2439, 24)


In [5]:
# lire les données dans prices_2025 pour recuperer les données de 2025 ( prices_df contient actuellement les données de 2024 seulement )

prices_folder_2025 = "data/prices_2025"          

# --- 2. Lire et concaténer tous les fichiers prices ---
price_files_2025 = glob(os.path.join(prices_folder_2025, "*.csv"))

prices_list_2025 = []#une liste de dataframes
for file in price_files_2025:
    # Lire le CSV en ignorant les lignes inutiles
    df = pd.read_csv(file, skiprows=[1, 2])  # ignore Ticker et Date,,,,
    
    # Renommer la première colonne si nécessaire
    if df.columns[0] != "Date":
        df.rename(columns={df.columns[0]: "Date"}, inplace=True)
    
    # Convertir la colonne Date en datetime
    df["Date"] = pd.to_datetime(df["Date"])
    
    # Ajouter le ticker depuis le nom du fichier
    ticker = os.path.basename(file).split(".")[0]
    df["Ticker"] = ticker
    
    prices_list_2025.append(df)

prices_df_2025 = pd.concat(prices_list_2025, ignore_index=True)
print("Prices dataset shape:", prices_df_2025.shape)

# 1. Supprimer la colonne 'Close' de prices_df_2025 si elle existe
if 'Close' in prices_df_2025.columns:
    prices_df_2025 = prices_df_2025.drop(columns=['Close'])

# 2. Filtrer pour ne garder que les dates >= 2024
prices_df_2025['Date'] = pd.to_datetime(prices_df_2025['Date'])
prices_df_2025 = prices_df_2025[prices_df_2025['Date'].dt.year >= 2024]

# 3. Concaténer avec prices_df
combined_prices = pd.concat([prices_df, prices_df_2025], ignore_index=True)

# 4. Supprimer les doublons basés sur Ticker + Date
combined_prices = combined_prices.drop_duplicates(subset=['Ticker', 'Date'], keep='first')

# 5. Trier par Ticker puis Date
combined_prices = combined_prices.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)

# 6. Mettre à jour prices_df
prices_df = combined_prices

print("Updated prices_df shape:", prices_df.shape)

#enregistrer le nouveau fichier CSV avec les données mises à jour
prices_df.to_csv("all_prices.csv", index=False)

#en effet maintenant chaque jour dans prices ( 2024 ou 2025) peut trouver le rapport fondamental le plus récent dans fundamentals_df (exemple: pour un rapport ds fundamentals mis dans decembre, il sera utilisé pour tous les jours de dec, jan et fev jusqu'au prochain rapport fondamental en mars)

Prices dataset shape: (2891473, 8)
Updated prices_df shape: (241824, 7)


In [6]:
#ajouter les indicateurs exploitables par le modele dans prices_df


# Normaliser le nom de la colonne ajustée
if 'Adj Close' in prices_df.columns:
    prices_df.rename(columns={'Adj Close': 'Close_Adj'}, inplace=True)
else:
    raise ValueError("No adjusted close column found")

print(prices_df[['Open', 'Close_Adj']].dtypes)
prices_df['Open'] = pd.to_numeric(prices_df['Open'], errors='coerce') # Convertir en numérique avec gestion des erreurs

# Daily return
prices_df['Daily_Return'] = (prices_df['Close_Adj'] - prices_df['Open']) / prices_df['Open'] * 100

# Volatility
prices_df['Volatility'] = prices_df.groupby('Ticker')['Daily_Return'].rolling(window=20).std().reset_index(0, drop=True)
prices_df['Annual_Volatility'] = prices_df['Volatility'] * np.sqrt(252)

# Moving Averages
for ma in [10, 50, 200]:
    prices_df[f'MA{ma}'] = prices_df.groupby('Ticker')['Close_Adj'].transform(lambda x: x.rolling(ma).mean())

# RSI
N = 14
def compute_rsi(x):
    delta = x.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(N).mean()
    avg_loss = loss.rolling(N).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

prices_df['RSI'] = prices_df.groupby('Ticker')['Close_Adj'].transform(compute_rsi)

# MACD
def compute_macd(x):
    ema12 = x.ewm(span=12, adjust=False).mean()
    ema26 = x.ewm(span=26, adjust=False).mean()
    return ema12 - ema26

prices_df['MACD'] = prices_df.groupby('Ticker')['Close_Adj'].transform(compute_macd)

# Drawdown
def compute_drawdown(x):
    roll_max = x.cummax()
    drawdown = (x - roll_max) / roll_max
    return drawdown

prices_df['Drawdown'] = prices_df.groupby('Ticker')['Close_Adj'].transform(compute_drawdown)

# Momentum features
for n in [1, 5, 10, 20]:
    prices_df[f'Momentum_{n}D'] = prices_df.groupby('Ticker')['Close_Adj'].transform(lambda x: x - x.shift(n))

#lag features
for n in [1, 5, 10]:
    prices_df[f'Return_Lag_{n}D'] = prices_df.groupby('Ticker')['Daily_Return'].shift(n)
#features temporaires / saisonnieres:
prices_df['Day_of_Week'] = prices_df['Date'].dt.dayofweek
prices_df['Month'] = prices_df['Date'].dt.month
prices_df['Quarter'] = prices_df['Date'].dt.quarter

#les features fond+prices:
# --- 1. Assurer que les dates sont du type datetime et sans NaN ---
prices_df['Date'] = pd.to_datetime(prices_df['Date'], errors='coerce')
fundamentals_df['Date'] = pd.to_datetime(fundamentals_df['Date'], errors='coerce')

# Supprimer les lignes où Date est NaT
prices_df = prices_df.dropna(subset=['Date'])
fundamentals_df = fundamentals_df.dropna(subset=['Date'])

# --- 2. Trier les DataFrames par Ticker puis Date ---
prices_df = prices_df.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)
fundamentals_df = fundamentals_df.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)



# Sauvegarder les données mises à jour
prices_df.to_csv("all_prices.csv", index=False)


Open          object
Close_Adj    float64
dtype: object


In [7]:

#fusionner les deux datasets prices et fundamentals
# Associer chaque ligne de prices avec le dernier rapport fondamental disponible

# Trier par Ticker puis Date si ce n'est pas déjà fait
prices_df = prices_df.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)
fundamentals_df = fundamentals_df.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)

# On merge "asof" par ticker (le rapport fondamental le plus récent <= date du prix)
merged_list = []
for ticker in prices_df['Ticker'].unique():
    prices_ticker = prices_df[prices_df['Ticker'] == ticker].copy()
    funds_ticker = fundamentals_df[fundamentals_df['Ticker'] == ticker].copy()
    
    if not funds_ticker.empty:
        merged = pd.merge_asof(
            prices_ticker,
            funds_ticker,
            on='Date',
            direction='backward',  # prendre le dernier rapport fondamental <= date
            tolerance=pd.Timedelta('92D')  # max 3 mois
        )
        merged_list.append(merged)
    else:
        merged_list.append(prices_ticker)

merged_df = pd.concat(merged_list, ignore_index=True)
print("Merged dataset shape:", merged_df.shape)

# Sauvegarder le dataset fusionné
merged_df.to_csv("merged_prices_fundamentals.csv", index=False)


Merged dataset shape: (241824, 50)


In [8]:
#nettoyer merged_df en supprimant les lignes avec des valeurs manquantes dans des colonnes critiques pour calculer des ratios financiers
# Colonnes critiques pour les ratios fondamentaux et basés sur les prix
critical_columns = [
    # Fondamentaux
    'Net Income Common Stockholders', 'Common Stock Equity', 'Net Income', 'Total Revenue',
    'Total Debt', 'Current Assets', 'Current Liabilities', 'Inventory', 'Operating Income',
    'Diluted EPS',
    # Prix
    'Close_Adj'
]

# Supprimer les lignes où l'une de ces colonnes est NaN
merged_df = merged_df.dropna(subset=critical_columns)

# --- Nettoyer les colonnes Ticker après merge ---
if 'Ticker_x' in merged_df.columns:
    merged_df.rename(columns={'Ticker_x': 'Ticker'}, inplace=True)

# Supprimer les doublons de colonnes Ticker
if 'Ticker_y' in merged_df.columns:
    merged_df.drop(columns=['Ticker_y'], inplace=True)

# Supprimer la deuxième colonne 'Ticker' si elle existe à la fin
merged_df = merged_df.loc[:, ~merged_df.columns.duplicated()]

# Vérifier qu'il ne reste qu'une seule colonne Ticker
print(merged_df.columns)


#enregistrer le dataset nettoyé
merged_df.to_csv("merged_prices_fundamentals.csv", index=False)

Index(['Ticker', 'Date', 'Close_Adj', 'High', 'Low', 'Open', 'Volume',
       'Daily_Return', 'Volatility', 'Annual_Volatility', 'MA10', 'MA50',
       'MA200', 'RSI', 'MACD', 'Drawdown', 'Momentum_1D', 'Momentum_5D',
       'Momentum_10D', 'Momentum_20D', 'Return_Lag_1D', 'Return_Lag_5D',
       'Return_Lag_10D', 'Day_of_Week', 'Month', 'Quarter', 'Total Revenue',
       'Operating Revenue', 'EBITDA', 'EBIT', 'Operating Income', 'Net Income',
       'Net Income From Continuing Operations',
       'Net Income Common Stockholders', 'Diluted Average Shares',
       'Basic Average Shares', 'Diluted EPS', 'Basic EPS', 'Total Debt',
       'Net Debt', 'Current Assets', 'Current Liabilities',
       'Cash Cash Equivalents And Short Term Investments',
       'Accounts Receivable', 'Inventory', 'Invested Capital',
       'Common Stock Equity', 'Cost Of Revenue'],
      dtype='object')


In [9]:


#calculer les ratios financiers dans le dataset fusionné merged_df

merged_df['ROE'] = merged_df['Net Income Common Stockholders'] / merged_df['Common Stock Equity']
merged_df['Profit_Margin'] = merged_df['Net Income'] / merged_df['Total Revenue']
merged_df['Debt_to_Equity'] = merged_df['Total Debt'] / merged_df['Common Stock Equity']

merged_df['EPS_Growth'] = merged_df.groupby('Ticker')['Diluted EPS'].pct_change()
merged_df['Revenue_Growth'] = merged_df.groupby('Ticker')['Total Revenue'].pct_change()
merged_df['Current_Ratio'] = merged_df['Current Assets'] / merged_df['Current Liabilities']
merged_df['Quick_Ratio'] = (merged_df['Current Assets'] - merged_df['Inventory']) / merged_df['Current Liabilities']
merged_df['Operating_Margin'] = merged_df['Operating Income'] / merged_df['Total Revenue']

# Pour les ratios basés sur les prix
merged_df['P_to_E'] = merged_df['Close_Adj'] / merged_df['Diluted EPS']
merged_df['Price_to_Book'] = merged_df['Close_Adj'] / merged_df['Common Stock Equity']
merged_df['Price_to_Revenue'] = merged_df['Close_Adj'] / merged_df['Total Revenue']



print(merged_df[['Ticker','Date','ROE','Profit_Margin','Debt_to_Equity','P_to_E']].head())


# Sauvegarder le dataset final avec les ratios financiers
merged_df.to_csv("merged_prices_fundamentals.csv", index=False)

    Ticker       Date       ROE  Profit_Margin  Debt_to_Equity      P_to_E
210      A 2024-10-31  0.059512       0.206349        0.574771  105.952053
211      A 2024-11-01  0.059512       0.206349        0.574771  111.285838
212      A 2024-11-04  0.059512       0.206349        0.574771  113.546178
213      A 2024-11-05  0.059512       0.206349        0.574771  114.164134
214      A 2024-11-06  0.059512       0.206349        0.574771  112.025739


In [10]:
#creer une table : ticker + les valeurs les plus recentes de chaque indicateur + target ( investir ou ne pas investir)
'''pourquoi la derniere valeur et nn pas la mouyenne ? 
Quand tu veux décider "investir ou pas", tu veux savoir l’état ACTUEL du titre :
Le RSI actuel
Le MACD actuel
La tendance actuelle (ma10, ma50, ma200)
Le drawdown actuel
Etc.
Les indicateurs techniques sont dynamiques.
Ils changent chaque jour.
Le signal d’investissement dépend de maintenant, pas d’une moyenne des derniers jours.
'''

# 1) Trier les données pour garantir l'ordre temporel
merged_df = merged_df.sort_values(["Ticker", "Date"])

# 2) Résumé par ticker : prendre la dernière ligne (la plus récente)
resume_df = merged_df.groupby("Ticker").tail(1).reset_index(drop=True)

#enregistrer le resume_df par ticker
resume_df.to_csv("ticker_summary.csv", index=False)

In [11]:
#afficher les colonnes de summary_df
print(resume_df.columns)
#ajouter une colonne target qui va etre calculée en fct de plusieurs criteres :
resume_df['target'] = np.nan
''' etre stricte dans les criteres d'investissement:
# --- 2. Fonction de signal d'investissement ---
def investment_signal(row, market_return=0.05):
    # Critères fondamentaux
    fundamentals_ok = (
        row['ROE'] > 0.15 and
        row['Profit_Margin'] > 0.1 and
        row['Debt_to_Equity'] < 1 and
        row['EPS_Growth'] > 0 and
        row['Revenue_Growth'] > 0 and
        row['P_to_E'] < 30 and
        row['Price_to_Book'] < 3 and
        row['Price_to_Revenue'] < 5
    )
    
    # Critères techniques
    technical_ok = (
        30 < row['RSI'] < 70 and
        row['Drawdown'] > -0.2 and
        row['Annual_Volatility'] < 0.5 and
        row['MACD'] > 0  # tendance haussière
    )
    
    # Comparaison avec le marché
    market_ok = (row['Daily_Return'] - market_return) > 0
    
    return 1 if fundamentals_ok and technical_ok and market_ok else 0

# --- 3. Appliquer le signal sur summary ---
resume_df['target'] = resume_df.apply(lambda row: investment_signal(row, market_return=0.05), axis=1)

ce qui est non flexible car target =0 toujours presque 
donc soyons plus souples dans les criteres cad plus realistes :
'''
def compute_investment_score(row):
    score = 0
    total = 0

    # Fondamentaux
    if 'ROE' in row: 
        score += int(row['ROE'] > 0.15)
        total += 1
    if 'Profit_Margin' in row:
        score += int(row['Profit_Margin'] > 0.1)
        total += 1
    if 'Debt_to_Equity' in row:
        score += int(row['Debt_to_Equity'] < 1)
        total += 1
    if 'EPS_Growth' in row:
        score += int(row['EPS_Growth'] > 0)
        total += 1
    if 'Revenue_Growth' in row:
        score += int(row['Revenue_Growth'] > 0)
        total += 1
    if 'P_to_E' in row:
        score += int(row['P_to_E'] < 30)
        total += 1
    if 'Price_to_Book' in row:
        score += int(row['Price_to_Book'] < 3)
        total += 1
    if 'Price_to_Revenue' in row:
        score += int(row['Price_to_Revenue'] < 5)
        total += 1

    # Prix / indicateurs techniques
    ticker_data = merged_df[merged_df['Ticker'] == row['Ticker']]
    
    if 'Volume' in row:
        if len(ticker_data) >= 20:
            mean_vol = ticker_data['Volume'].iloc[-20:].mean()
            score += int(row['Volume'] > mean_vol)
        else:
            score += 1  # si moins de 20 jours, on considère OK
        total += 1

    if 'RSI' in row:
        score += int(30 < row['RSI'] < 70)
        total += 1
    if 'Drawdown' in row:
        score += int(row['Drawdown'] > -0.2)
        total += 1
    if 'Annual_Volatility' in row:
        score += int(row['Annual_Volatility'] < 0.5)
        total += 1
    if 'MACD' in row:
        score += int(row['MACD'] > 0)  # tendance positive
        total += 1

    return score / total if total > 0 else 0


resume_df['Invest_Score'] = resume_df.apply(compute_investment_score, axis=1)#calculer le score d'investissement pour chaque ligne cad pour chaque ticker

# --- 3. Définir le target selon un seuil (ex: 70%) ---
threshold = 0.5#içi on peut ajuster le seuil en fonction de la rigueur souhaitée et de combien on souhaite etre strict dans les criteres
resume_df['target'] = (resume_df['Invest_Score'] >= threshold).astype(int)

print(resume_df[['Ticker', 'target']].head())
# Sauvegarder le résumé final avec la cible d'investissement
resume_df.to_csv("ticker_summary.csv", index=False)


Index(['Ticker', 'Date', 'Close_Adj', 'High', 'Low', 'Open', 'Volume',
       'Daily_Return', 'Volatility', 'Annual_Volatility', 'MA10', 'MA50',
       'MA200', 'RSI', 'MACD', 'Drawdown', 'Momentum_1D', 'Momentum_5D',
       'Momentum_10D', 'Momentum_20D', 'Return_Lag_1D', 'Return_Lag_5D',
       'Return_Lag_10D', 'Day_of_Week', 'Month', 'Quarter', 'Total Revenue',
       'Operating Revenue', 'EBITDA', 'EBIT', 'Operating Income', 'Net Income',
       'Net Income From Continuing Operations',
       'Net Income Common Stockholders', 'Diluted Average Shares',
       'Basic Average Shares', 'Diluted EPS', 'Basic EPS', 'Total Debt',
       'Net Debt', 'Current Assets', 'Current Liabilities',
       'Cash Cash Equivalents And Short Term Investments',
       'Accounts Receivable', 'Inventory', 'Invested Capital',
       'Common Stock Equity', 'Cost Of Revenue', 'ROE', 'Profit_Margin',
       'Debt_to_Equity', 'EPS_Growth', 'Revenue_Growth', 'Current_Ratio',
       'Quick_Ratio', 'Operating_Ma

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier #model basé sur l'arbre de décision 
from sklearn.metrics import classification_report

# 1. Liste complète des features
features = [
    # Fondamentaux
    'ROE', 'Profit_Margin', 'Debt_to_Equity', 'EPS_Growth', 'Revenue_Growth',
    'Current_Ratio', 'Quick_Ratio', 'Operating_Margin',
    'P_to_E', 'Price_to_Book', 'Price_to_Revenue',
    
    # Prix / techniques
    'Daily_Return', 'Volatility', 'Annual_Volatility',
    'MA10', 'MA50', 'MA200', 'RSI', 'MACD', 'Drawdown',
    'Momentum_1D', 'Momentum_5D', 'Momentum_10D', 'Momentum_20D',
    'Return_Lag_1D', 'Return_Lag_5D', 'Return_Lag_10D',
    
    # Variables temporelles (optionnel)
    'Day_of_Week', 'Month', 'Quarter',
    
    # Score d'investissement (optionnel)
    'Invest_Score'
]
#supprimer la colonne Invest_Score:
features = [f for f in features if f != 'Invest_Score']
'''
Inclure 'Invest_Score' entraînerait un data leakage :
le modèle apprendrait directement la relation (>0.5 → target=1) au lieu d'apprendre
les vrais patterns des features, ce qui biaiserait l'apprentissage et
réduirait sa capacité à généraliser.
le modele ne fait pas de calculs, il apprend a partir des features qui le sont fournis brutement, il ne peut mm pas calculer les ratios, 
donc je dois fournir ces données et il doit apprendre a partir de ça sans tricher(guessing that >0.5 means you can invest).
score > 0.5 est calculée juste pour pouvoir remplir le target de manière réaliste.
'''

# 2. Supprimer lignes avec NaN
ml_df = resume_df.dropna(subset=features + ['target']).copy()
X = ml_df[features]
y = ml_df['target']

# 3. Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y    #20% des données pour le test et 80% pour l'entrainement
)

# 4. Standardisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Modèle
model = RandomForestClassifier(
    n_estimators=200,#nbr d'arbres dans la foret , plus il ya d'arbres plus le modele est robuste mais plus lent a entrainer
    max_depth=8,#profondeur maximale de chaque arbre pour eviter le surapprentissage
    random_state=42, #c
    class_weight='balanced'#si les classes sont déséquilibrées, cela aide à compenser le déséquilibre
)
model.fit(X_train_scaled, y_train)#entrainer le modele sur les entrées standardisées pour predire la target y

# 6. Évaluation
y_pred = model.predict(X_test_scaled)#predire la target sur les données de test 
print(classification_report(y_test, y_pred))

# 7. Sauvegarder le modèle et le scaler pour utilisation future
import joblib
joblib.dump(model, 'investment_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(features, 'model_features.pkl')
print("Modèle, scaler et features sauvegardés!")

'''
Train set (X_train, y_train) :

X_train contient les features.
y_train contient les valeurs réelles de target.
Le modèle apprend en comparant sa prédiction avec y_train et ajuste ses paramètres (c’est l’apprentissage supervisé).

Test set (X_test, y_test) :

X_test contient les features.
y_test contient aussi les valeurs réelles de target, mais le modèle ne les voit jamais pendant l’entraînement.
On utilise y_test seulement après la prédiction pour vérifier la performance du modèle (y_pred) via précision, rappel, F1-score, etc.
pour ce faire on utilise classification_report qui compare y_test (vraies valeurs) avec y_pred (prédictions du modèle) et affiche des métriques d’évaluation!



'''
#forest est beaucoup mieux pour predire la classe 0=>Éviter les faux investissements → Random Forest.

              precision    recall  f1-score   support

           0       0.83      0.98      0.90        49
           1       0.89      0.44      0.59        18

    accuracy                           0.84        67
   macro avg       0.86      0.71      0.74        67
weighted avg       0.84      0.84      0.82        67

Modèle, scaler et features sauvegardés!


'\nTrain set (X_train, y_train) :\n\nX_train contient les features.\ny_train contient les valeurs réelles de target.\nLe modèle apprend en comparant sa prédiction avec y_train et ajuste ses paramètres (c’est l’apprentissage supervisé).\n\nTest set (X_test, y_test) :\n\nX_test contient les features.\ny_test contient aussi les valeurs réelles de target, mais le modèle ne les voit jamais pendant l’entraînement.\nOn utilise y_test seulement après la prédiction pour vérifier la performance du modèle (y_pred) via précision, rappel, F1-score, etc.\npour ce faire on utilise classification_report qui compare y_test (vraies valeurs) avec y_pred (prédictions du modèle) et affiche des métriques d’évaluation!\n\n\n\n'

In [18]:
#essayer un autre modele: XGBoost

from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=49/18  # pour gérer le déséquilibre des classes
)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

#XGBoost est beaucoup mieux pour predire la classe 1 => Maximiser la détection des bons investissements → XGBoost.

              precision    recall  f1-score   support

           0       0.88      0.86      0.87        49
           1       0.63      0.67      0.65        18

    accuracy                           0.81        67
   macro avg       0.75      0.76      0.76        67
weighted avg       0.81      0.81      0.81        67



In [19]:
from sklearn.svm import SVC

model = SVC(kernel='rbf', class_weight='balanced', probability=True)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(classification_report(y_test, y_pred))
#moins solide que les deux autres modèles'''

              precision    recall  f1-score   support

           0       0.87      0.69      0.77        49
           1       0.46      0.72      0.57        18

    accuracy                           0.70        67
   macro avg       0.67      0.71      0.67        67
weighted avg       0.76      0.70      0.72        67



In [20]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(classification_report(y_test, y_pred))
#RF ET XGBOOST restent les meilleurs



              precision    recall  f1-score   support

           0       0.89      0.69      0.78        49
           1       0.48      0.78      0.60        18

    accuracy                           0.72        67
   macro avg       0.69      0.74      0.69        67
weighted avg       0.78      0.72      0.73        67



In [21]:
#essayons une combinaison de xgboost et random forest via un Voting Classifier

from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight='balanced')
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', scale_pos_weight=2)

ensemble = VotingClassifier(estimators=[('rf', rf), ('xgb', xgb)], voting='soft')  # 'soft' = moyenne des probabilités
ensemble.fit(X_train_scaled, y_train)

y_pred = ensemble.predict(X_test_scaled)
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

#Forest reste le meilleur choix 


              precision    recall  f1-score   support

           0       0.83      0.90      0.86        49
           1       0.64      0.50      0.56        18

    accuracy                           0.79        67
   macro avg       0.74      0.70      0.71        67
weighted avg       0.78      0.79      0.78        67



c:\Users\malek\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [19:39:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [17]:
print(scaler.feature_names_in_)

['ROE' 'Profit_Margin' 'Debt_to_Equity' 'EPS_Growth' 'Revenue_Growth'
 'Current_Ratio' 'Quick_Ratio' 'Operating_Margin' 'P_to_E' 'Price_to_Book'
 'Price_to_Revenue' 'Daily_Return' 'Volatility' 'Annual_Volatility' 'MA10'
 'MA50' 'MA200' 'RSI' 'MACD' 'Drawdown' 'Momentum_1D' 'Momentum_5D'
 'Momentum_10D' 'Momentum_20D' 'Return_Lag_1D' 'Return_Lag_5D'
 'Return_Lag_10D' 'Day_of_Week' 'Month' 'Quarter']
